## Runge-Kutta Methods
1. [Runge–Kutta methods](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods)
2. [List of Runge–Kutta methods](https://en.wikipedia.org/wiki/List_of_Runge%E2%80%93Kutta_methods)
Butcher tableau for an **explicit** Runge-Kutta scheme 
\begin{array}
{c|ccccc}
0\\
c_2 & a_{21}\\
c_3 & a_{31} &a_{32} \\
\vdots & \vdots & & \ddots\\
c_s& a_{s1}& a_{s2} & \cdots & a_{s,s-1}\\
\hline
& b_1 &b_2 &\cdots & b_{s-1} & b_s \\ 
\end{array}
Then
$$
    y_{n+1} = y_n+h\sum_{i=1}^{s}b_i k_i
$$
where
\begin{align}
    k_1 &= f(t_n,y_n),\cr
    k_2 &= f(t_n+c_2h,y_n+h(a_{21}k_1)),\cr
    k_3 &= f(t_n+c_3h,y_n+h(a_{31}k_1+a_{32}k_2)),\cr
    &\vdots\cr
    k_s &= f(t_n+c_sh,y_n+h(a_{s1}k_1+a_{s2}k_2+\cdots+a_{s,s-1}k_{s-1}))\cr
\end{align}

## General Purpose Explicit Runge-Kutta Method

### Imports

In [1]:
import numpy as np, numpy.linalg as LA, math
from math import pi as π
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp

In [2]:
def rk(fun,t_span,y0,n,tableau=None,verbose=False):
    """
    fun: vector field function for ODE
    t_span: array of length 2 specify the start and end time
    y0: array specifying an initial condition
    n: number of fixed steps to take
    tableau: tuple (a,b,c) describing the Butcher Tableau for the method
    """
    y0 = np.array(y0)
    t = np.linspace(t_span[0],t_span[1],n+1)
    h = (t_span[1]-t_span[0])/n
    y = np.empty((n+1,y0.size))
    y[0] = y0
    if tableau is None:
        s = 4
        a = np.array([[1/2,0,0],[0,1/2,0],[0,0,1]])
        b = np.array([1/6,1/3,1/3,1/6])
        c = np.array([0,1/2,1/2,1])
    else:
        a,b,c = tableau
        s = len(b)
    if verbose:
        print('s = %d' %s)
        print('n = %d' %n)
        print('h = %.4f' %h)
        print('a:',a)
        print('b:',b)
        print('c:',c)
    K = np.empty(s)
    for i in range(n):
        K[0] = fun(t[i]+c[0]*h,y[i].squeeze())
        for j in range(1,s):
            ttmp = t[i] + c[j]*h
            ytmp = y[i] + h*K[:j]@a[j-1,:j]
            K[j] = fun(ttmp,ytmp.squeeze())
        y[i+1] = y[i] + h * b@K
    return t, y

### Bogacki–Shampine

In [3]:
a = np.array([[1/2,0,0],[0,3/4,0],[2/9,1/3,4/9]])
b = np.array([2/9,1/3,4/9,0])
c = np.array([0,1/2,3/4,1])

b1 = np.array([7/24,1/4,1/3,1/8])

In [4]:
tab3 = (a,b,c)
tab2 = (a,b1,c)

In [5]:
fun = lambda t, y: -2.*y
t0,t1 = 0,0.1
y0 = 1

nsteps = 1
h = (t1-t0)/nsteps

t_span = [t0,t1]

In [6]:
tout,yout = rk(fun,t_span,y0,nsteps,tableau=tab3)
yout = yout.squeeze()
math.exp(-2.*t1)-yout[-1],(math.exp(-2.*t1)-yout[-1])/h**4

(np.float64(6.408641131516735e-05), np.float64(0.6408641131516734))

In [7]:
t0,t1 = 0,0.05
h = (t1-t0)/nsteps
t_span = [t0,t1]
tout,yout = rk(fun,t_span,y0,nsteps,tableau=tab3)
yout = yout.squeeze()
math.exp(-2.*t1)-yout[-1],(math.exp(-2.*t1)-yout[-1])/h**4

(np.float64(4.084702626139247e-06), np.float64(0.6535524201822794))

In [8]:
t0,t1 = 0,0.1
h = (t1-t0)/nsteps
t_span = [t0,t1]
tout,yout = rk(fun,t_span,y0,nsteps,tableau=tab2)
yout = yout.squeeze()
math.exp(-2.*t1)-yout[-1],(math.exp(-2.*t1)-yout[-1])/h**3

(np.float64(0.000197419744648486), np.float64(0.19741974464848594))

In [9]:
t0,t1 = 0,0.05
h = (t1-t0)/nsteps
t_span = [t0,t1]
tout,yout = rk(fun,t_span,y0,nsteps,tableau=tab2)
yout = yout.squeeze()
math.exp(-2.*t1)-yout[-1],(math.exp(-2.*t1)-yout[-1])/h**3

(np.float64(2.283470262620657e-05), np.float64(0.1826776210096525))

In [10]:
t0,t1 = 0,0.1
h = (t1-t0)/nsteps
t_span = [t0,t1]
tout3,yout3 = rk(fun,t_span,y0,nsteps,tableau=tab3)
tout2,yout2 = rk(fun,t_span,y0,nsteps,tableau=tab2)

yout2 = yout2.squeeze()
yout3 = yout3.squeeze()

yout2[-1]-yout3[-1]

np.float64(-0.00013333333333331865)

In [12]:
yout2

array([1.        , 0.81853333])

In [14]:
tol = 1e-8
h = tol/(yout2[-1]-yout3[-1])
h

np.float64(-7.500000000000826e-05)

In [16]:
t0,t1 = 0,7.500000000000826e-05
h = (t1-t0)/nsteps
t_span = [t0,t1]
tout,yout = rk(fun,t_span,y0,nsteps,tableau=tab3)
yout = yout.squeeze()
math.exp(-2.*t1)-yout[-1],(math.exp(-2.*t1)-yout[-1])/h**4

(np.float64(0.0), np.float64(0.0))

In [21]:
t0,t1 = 0,1
atol,rtol = 5e-9,5e-9
t_span = [t0,t1]
sol = solve_ivp(fun,t_span,[y0],method='RK23',atol=atol,rtol=rtol)
sol

  message: The solver successfully reached the end of the integration interval.
  success: True
   status: 0
        t: [ 0.000e+00  2.924e-04 ...  9.962e-01  1.000e+00]
        y: [[ 1.000e+00  9.994e-01 ...  1.364e-01  1.353e-01]]
      sol: None
 t_events: None
 y_events: None
     nfev: 698
     njev: 0
      nlu: 0